教学演示 思维链

核心学习目标

掌握任务语义解析、事实状态推理、信息完备性分析，理解复杂任务中的显式推理链构建机制

设计

输入：用户请求
展示使用思维链推理出 已知、查询可知、需向用户澄清事实等

**本课两个重点**：
1. 用 **LangChain `PromptTemplate`** 生成与 `str.format(task=…)` 相同的完整提示词，并**展示全文**（教学演示「思维链提示词长什么样」）。
2. （可选）用 **`ChatOpenAI`** 发一条请求，看模型按四段标题（已给出或已验证的事实 / 需要查阅的事实 / 需要推导的事实 / 有根据的猜测）回复。


## 0. 环境

```bash
pip install langchain-core langchain-openai httpx
```

若最后一格要真调模型，请设置：`OPENAI_API_KEY`、`OPENAI_BASE_URL`、`OPENAI_MODEL`（或在代码里临时写死）。

In [6]:
# Windows 下避免 print 长中文/特殊符号时报编码错（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

## 1. 思维链提示词模板

占位符只有一个：`{task}` → 替换为用户的自然语言任务。

In [7]:
FACTS_PROMPT = """在我提出正式请求之前，先认真回答以下预调查问题，务必做到知无不言、言无不尽。你具备最顶尖的常识问答水平和大师级的谜题破解能力，知识体系完整且深厚，请毫无保留地运用你的全部知识储备。

用户输入任务：

{task}

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。
"""

# 课堂示例任务
task = """你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略"""

## 2. 用 LangChain 生成「完整提示词」

`PromptTemplate.from_template` 与 Python 的 `FACTS_PROMPT.format(task=…)` 在这里**等价**，便于后面接 LCEL、链式调试等。

In [8]:
from langchain_core.prompts import PromptTemplate

facts_template = PromptTemplate.from_template(FACTS_PROMPT)
full_prompt = facts_template.format(task=task.strip())


assert full_prompt == FACTS_PROMPT.format(task=task.strip())

## 3. 展示：拼好后的整条「思维链」提示词

下面就是编排器第一步要发给模型的**完整用户消息**（节选可在课堂改成 `[:2000]` 等）。

In [9]:
from IPython.display import display, Markdown

display(Markdown(f"**字符数：** {len(full_prompt)}"))
display(Markdown("```text\n" + full_prompt + "\n```"))

**字符数：** 517

```text
在我提出正式请求之前，先认真回答以下预调查问题，务必做到知无不言、言无不尽。你具备最顶尖的常识问答水平和大师级的谜题破解能力，知识体系完整且深厚，请毫无保留地运用你的全部知识储备。

用户输入任务：

你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。

```

## 4.（可选）LangChain `ChatOpenAI` 调用一次

把上格生成的 `full_prompt` 作为**一条用户消息**发给 OpenAI 兼容网关即可。下面用环境变量；课堂演示也可改成与你 `ChatOpenAI(model=..., base_url=..., api_key=...)` 一样的字面量。

In [10]:
import os

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

kw = dict(
        model="qwen",
        temperature=0,
        base_url="model base_url",
        api_key="your api key ",
)
import httpx
kw["http_client"] = httpx.Client(verify=False)
llm = ChatOpenAI(**kw)
resp = llm.invoke([HumanMessage(content=full_prompt)])
display(Markdown("### 模型输出"))
display(Markdown(resp.content if isinstance(resp.content, str) else str(resp.content)))

### 模型输出

**1. 已给出或已验证的事实**  
- 旅行时间为“五一假期”。  
- 目的地包括上海、苏州、杭州三个城市。  
- 需求包括行程路线、酒店推荐、天气情况和美食攻略。  

**2. 需要查阅的事实**  
- 2026 年 5 月初上海、苏州、杭州的具体天气预报（气温、降雨概率、风力等）。  
- 各城市主要景点的开放时间、是否需要提前预约以及门票信息。  
- 高铁/动车、长途巴士等城市间的最新时刻表及票价。  
- 2026 年 5 月的酒店房价、可用房型及用户评价（可通过携程、Booking、Airbnb 等平台获取）。  
- 当地最新的防疫或其他旅行限制政策（如有）。  
- 各城市的热门美食店铺地址、营业时间及人均消费（可参考大众点评、美食博客等）。  

**3. 需要推导的事实**  
- 基于城市间距离和交通方式，推算出最省时、最省钱的行程顺序及每日行程时长。  
- 计算从上海出发至苏州、再至杭州的预计交通时间（包括候车、乘车、到站后市内交通）。  
- 估算整体旅行预算（交通、住宿、餐饮、门票等），并给出不同档次的预算区间。  
- 根据历史气象数据，推断五一期间该地区的平均气温、降雨概率，以补充天气预报的不足。  

**4. 有根据的猜测**  
- 2026 年五一期间，上海、苏州、杭州的日均气温大约在 20 °C–25 °C 左右，可能会有短时阵雨。  
- 最合理的行程顺序可能为：上海 → 苏州（约 30 km，乘坐高铁约 30 min） → 杭州（苏州至杭州约 150 km，乘坐高铁约 1 h） → 返回上海（杭州至上海约 180 km，乘坐高铁约 1 h）。  
- 推荐的中高端酒店可能包括：上海外滩茂悦大酒店、苏州金鸡湖凯悦酒店、杭州西湖国宾馆等。  
- 当地必尝美食猜测：上海小笼包、生煎、糟鸭；苏州松鼠桂鱼、碧螺春茶点；杭州西湖醋鱼、龙井虾仁、叫化鸡。  
- 预计每日行程安排可在 8–10 小时内完成，留出足够时间品尝美食和休息。

## 5. 小结

- **思维链提示词**在这里 = 固定说明 + 嵌入的 `{task}` + 要求模型按四段标题作答；**全部放在一条 user 内容里**。
- **LangChain**：`PromptTemplate` 负责字符串拼装；`ChatOpenAI` 负责调用；